In [6]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [7]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [8]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [9]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_12_0_23,1.000000,0.645479,1.00000,1.0,1.000000,1.036166e-10,0.210459,3.865061e-10,2.476980e-10,3.171020e-10,0.000003,0.000010,1.000000,0.000011,183.980647,268.083079,"Hidden Size=[17], regularizer=0.3, learning_ra..."
1,model_12_0_21,1.000000,0.645495,1.00000,1.0,1.000000,1.083207e-10,0.210449,3.349204e-10,1.418687e-10,2.383946e-10,0.000004,0.000010,1.000000,0.000011,183.891851,267.994282,"Hidden Size=[17], regularizer=0.3, learning_ra..."
2,model_12_0_22,1.000000,0.645483,1.00000,1.0,1.000000,1.104478e-10,0.210457,4.104238e-10,2.338679e-10,3.221458e-10,0.000004,0.000011,1.000000,0.000011,183.852955,267.955387,"Hidden Size=[17], regularizer=0.3, learning_ra..."
3,model_12_3_7,1.000000,0.645496,1.00000,1.0,1.000000,2.362307e-10,0.210449,8.652346e-10,5.281551e-14,4.326406e-10,0.000004,0.000015,1.000000,0.000016,182.332424,266.434856,"Hidden Size=[17], regularizer=0.3, learning_ra..."
4,model_12_3_5,1.000000,0.645496,1.00000,1.0,1.000000,2.362307e-10,0.210449,8.652346e-10,5.281551e-14,4.326406e-10,0.000004,0.000015,1.000000,0.000016,182.332424,266.434856,"Hidden Size=[17], regularizer=0.3, learning_ra..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,model_24_5_21,0.999906,0.583994,1.00000,1.0,1.000000,5.589672e-05,0.246959,6.891060e-15,4.238383e-14,2.672447e-14,0.001692,0.007476,1.000040,0.007795,181.584010,280.312951,"Hidden Size=[20], regularizer=0.3, learning_ra..."
1996,model_24_5_2,0.999906,0.583994,1.00000,1.0,1.000000,5.589672e-05,0.246959,6.891060e-15,4.238383e-14,2.672447e-14,0.001692,0.007476,1.000040,0.007795,181.584010,280.312951,"Hidden Size=[20], regularizer=0.3, learning_ra..."
1997,model_24_5_14,0.999906,0.583994,1.00000,1.0,1.000000,5.589672e-05,0.246959,6.891060e-15,4.238383e-14,2.672447e-14,0.001692,0.007476,1.000040,0.007795,181.584010,280.312951,"Hidden Size=[20], regularizer=0.3, learning_ra..."
2061,model_19_5_1,0.999854,0.638977,0.99999,1.0,0.999993,8.641661e-05,0.214319,9.476429e-06,8.351771e-12,4.738219e-06,0.001577,0.009296,1.000066,0.009692,172.712661,266.566100,"Hidden Size=[19], regularizer=0.3, learning_ra..."
